Objective

This notebook demonstrates the capability of the NAV-POC modules to analyze ArduPilot NAV domain behavior during a mission run, including normal and anomaly conditions.

Goal:

Show how NAV pipeline processes mission logs
Present GPS behavior during normal and degraded conditions
Demonstrate how anomalies appear in NAV domain outputs
Link raw logs to structured mission-level views

In [3]:
import os
import duckdb
import pandas as pd

DB_PATH = os.path.abspath("../bin/vault/warehouse_df/NAV_20260319_1332_parts/nav_master.duckdb")
con = duckdb.connect(DB_PATH)

print("✅ Connected to DuckDB")
print(con.execute("SHOW TABLES").fetchall())

✅ Connected to DuckDB
[('com_master',), ('context_master',), ('est_master',), ('fmt_master',), ('label_master',), ('mission_stats',), ('msg_type_master',), ('nav_ai_assistance',), ('nav_context',), ('nav_flight_context',), ('nav_master',), ('nav_mot_state_timeline',), ('nav_motion',), ('nav_rca_context',), ('nav_rca_gps_mot',), ('nav_sig_state_timeline',), ('nav_state_fc',), ('nav_state_master',), ('nav_state_timeline',), ('nav_windows',), ('power_master',), ('rule_master',), ('sys_master',)]


Step 1: Check GPS data availability
This step checks whether GPS data exists in the NAV dataset before further analysis.

If no GPS data is found, the mission cannot be analyzed for GPS behavior.

In [4]:
anchor_df = con.execute("""
SELECT msg_type, COUNT(*)
FROM nav_master
WHERE msg_type = 'GPS'
GROUP BY msg_type;""").df()

print("✅ Anchor loaded")
display(anchor_df)

✅ Anchor loaded


,msg_type,count_star()
0,GPS,2205


Step 2: Check GPS data fields
This step checks if key GPS values like satellite count (NSats) and status are present in the logs.

In [5]:
integrity_df = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(NSats) AS nsats_present,
    COUNT(Status) AS status_present
FROM nav_master
WHERE msg_type = 'GPS';
""").df()

print("✅ GPS Parameter Integrity Check")
display(integrity_df)

✅ GPS Parameter Integrity Check


,total_rows,nsats_present,status_present
0,2205,2205,2205


Step 3: GPS fix types

This step shows the different GPS fix states observed during the mission.

In [6]:
status_dist_df = con.execute("""
SELECT Status, COUNT(*) AS count
FROM nav_master
WHERE msg_type = 'GPS'
GROUP BY Status
ORDER BY Status;
""").df()

print("✅ GPS Status Distribution")
display(status_dist_df)

✅ GPS Status Distribution


,Status,count
0,1.0,1
1,6.0,2204


Step 4: Satellite count over time

This step shows how GPS satellite visibility changes during the mission.

In [7]:
nsats_dist_df = con.execute("""
SELECT NSats, COUNT(*) AS count
FROM nav_master
WHERE msg_type = 'GPS'
GROUP BY NSats
ORDER BY NSats;
""").df()

print("✅ NSats Distribution")
display(nsats_dist_df)

✅ NSats Distribution


,NSats,count
0,0.0,1301
1,3.0,1
2,7.0,56
3,12.0,847


Step 5: Compare GPS sensor and flight controller state

This step compares GPS sensor values (NSats) with flight controller reported status to understand how they relate during the mission.

In [8]:
mismatch_df = con.execute("""
SELECT
    TimeUS,
    CAST(NSats AS INTEGER) AS NSats,
    CAST(Status AS INTEGER) AS Status,

    CASE
        WHEN NSats = 0 THEN 'LOSS'
        WHEN NSats < 8 THEN 'DEGRADED'
        ELSE 'HEALTHY'
    END AS sensor_state,

    CASE
        WHEN Status <= 1 THEN 'LOSS'
        WHEN Status >= 3 THEN 'HEALTHY'
        ELSE 'DEGRADED'
    END AS fc_state,

    CASE
        WHEN
            (NSats = 0 AND Status >= 3) OR
            (NSats >= 8 AND Status <= 1)
        THEN 'MISMATCH'
        ELSE 'ALIGNED'
    END AS integrity_flag

FROM nav_master
WHERE msg_type = 'GPS'
ORDER BY TimeUS
LIMIT 100;
""").df()

print("✅ Sensor vs FC Consistency Check")
display(mismatch_df)

✅ Sensor vs FC Consistency Check


,TimeUS,NSats,Status,sensor_state,fc_state,integrity_flag
0,9819404,3,1,DEGRADED,LOSS,ALIGNED
1,9999332,0,6,LOSS,HEALTHY,MISMATCH
2,10199252,0,6,LOSS,HEALTHY,MISMATCH
3,10399172,0,6,LOSS,HEALTHY,MISMATCH
4,10599092,0,6,LOSS,HEALTHY,MISMATCH
...,...,...,...,...,...,...
95,28799309,0,6,LOSS,HEALTHY,MISMATCH
96,28999229,0,6,LOSS,HEALTHY,MISMATCH
97,29199149,0,6,LOSS,HEALTHY,MISMATCH
98,29399069,0,6,LOSS,HEALTHY,MISMATCH


Step 6: GPS behavior over time

This step groups GPS behavior into time windows to show how it changes during the mission.

In [9]:
timeline_df = con.execute("""
WITH base AS (
    SELECT
        TimeUS,
        CAST(NSats AS INTEGER) AS nsats,
        CAST(Status AS INTEGER) AS status
    FROM nav_master
    WHERE msg_type = 'GPS'
),

grouped AS (
    SELECT *,
        ROW_NUMBER() OVER (ORDER BY TimeUS) -
        ROW_NUMBER() OVER (
            PARTITION BY nsats, status
            ORDER BY TimeUS
        ) AS grp
    FROM base
)

SELECT
    MIN(TimeUS) AS start_time,
    MAX(TimeUS) AS end_time,
    ROUND((MAX(TimeUS) - MIN(TimeUS)) / 1000000.0, 2) AS duration_sec,
    nsats AS nsats_value,
    status AS status_value

FROM grouped
GROUP BY grp, nsats, status
HAVING duration_sec > 0.1
ORDER BY start_time;
""").df()

print("✅ Mission Timeline: NSats vs Status")
display(timeline_df)

✅ Mission Timeline: NSats vs Status


,start_time,end_time,duration_sec,nsats_value,status_value
0,9999332,258999692,249.0,0,6
1,259199612,405799282,146.6,12,6
2,405999202,416999800,11.0,7,6
3,417199720,427999565,10.8,0,6
4,428199485,450599688,22.4,12,6


Mission Log Summary

This section shows how GPS sensor values (NSats) and flight controller status (Status) behave over time during the mission.

Investigation Steps

The following checks are used to understand GPS behavior during the selected time window.

Check sensor values
Look at NSats values during the selected time period.
Check flight controller status
Look at Status values during the same time period.
Compare both signals
Observe how NSats and Status change over time.
Check EKF values (if available)
Review EKF innovation values during the same period.

In [11]:
# Query to check EKF innovations during selected time window
ekf_check_df = con.execute(f"""
    SELECT
        TimeUS,
        msg_type,
        IVN, -- GPS North Velocity Innovation
        IVE, -- GPS East Velocity Innovation
        IVD, -- GPS Down Velocity Innovation
        IPN, -- GPS North Position Innovation
        IPE  -- GPS East Position Innovation
    FROM nav_master
    WHERE msg_type IN ('NKF3', 'XKF3')
    AND TimeUS BETWEEN 9999332 AND 258999692 -- Focus on Window 0
    ORDER BY TimeUS
""").df()

display(ekf_check_df)

,TimeUS,msg_type,IVN,IVE,IVD,IPN,IPE
0,9999332,XKF3,0.0,0.0,0.0,0.00,0.00
1,9999332,XKF3,0.0,0.0,0.0,0.00,0.00
2,10039316,XKF3,0.0,0.0,0.0,0.00,0.00
3,10039316,XKF3,0.0,0.0,0.0,0.00,0.00
4,10079300,XKF3,0.0,0.0,0.0,0.00,0.00
...,...,...,...,...,...,...,...
12447,258919724,XKF3,0.0,0.0,0.0,-0.03,0.04
12448,258959708,XKF3,0.0,0.0,0.0,-0.03,0.04
12449,258959708,XKF3,0.0,0.0,0.0,-0.03,0.04
12450,258999692,XKF3,0.0,0.0,0.0,-0.03,0.04


EKF innovation values are shown for the selected time window.

Tests

The following tests were performed using SITL-generated logs:

Test A (Signal change):

Satellite count (NSats) was reduced.
The reported GPS status remained at 6 during this period.

Test B (Hardware change):

GPS was disabled.
The reported status changed to 0.

Observation

During the test window, changes in satellite count did not immediately reflect in the reported GPS status.

A status change was observed only when the GPS source itself was disabled.

Note: Observations are based on DataFlash logs generated in SITL.